# Classifying CIFAR-10 with Pretrained CLIP: Original vs Reconstructed Images


- [View Solution Notebook](./solutions.html)
- [View Project Page](https://www.codecademy.com/content-items/3852ed8e56994723932d023c09e24f76)

## Objective

In this project, you'll explore how the **image reconstruction quality** of autoencoders affects the zero-shot classification performance of a pretrained CLIP multimodal model on the CIFAR-10 dataset. 

#### CIFAR-10 Dataset

[CIFAR-10](https://www.cs.toronto.edu/~kriz/cifar.html) is a widely used dataset for computer vision tasks, containing 60,000 color images where each image is:
- **32x32** pixels
- **3** color channels (RGB)
- Labeled into **1 of 10 object classes**

Here's a preview of each of the 10 classes:

<p style="text-align: center;">
  <img src="cifar10_preview.png" width="60%">
</p>

```py
cifar10_classes = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
               'dog', 'frog', 'horse', 'ship', 'truck']
```

Specifically, you will:

**A.** Build and train a convolution-based autoencoder to compress CIFAR-10 images into a low-dimensional latent representation and then reconstruct them to extract meaningful features. 

**B.** Perform zero-shot classification with a pretrained CLIP model to classify both the _original_ and _reconstructed_ images, and compare the side-by-side performance.

Let's get started!

---

### Setup: Import libraries

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from transformers import CLIPProcessor, CLIPModel
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Task Group 1: Setup and Loading

#### CIFAR-10 Dataset

[CIFAR-10](https://www.cs.toronto.edu/~kriz/cifar.html) is a widely used dataset for computer vision tasks, containing 60,000 color images where each image is:
- **32x32** pixels
- **3** color channels (RGB)
- Labeled into **1 of 10 object classes**

The 10 classes are: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck.

### Task 1: Define CIFAR-10 class names

Create a list that contains each of the CIFAR-10 class names.

In [ ]:
cifar10_classes = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
               'dog', 'frog', 'horse', 'ship', 'truck']

### Task 2: Load the testing set

Load the CIFAR-10 testing set, which contains 10,000 images. Apply a simple preprocessing pipeline that converts each image into a PyTorch tensor.

The CIFAR-10 dataset is located in the following directory: `'/home/ccuser/data'`.

In [ ]:
data_dir = '/home/ccuser/data'

transform = transforms.Compose([
    transforms.ToTensor(),
])

test_data = datasets.CIFAR10(root=data_dir, 
                              train=False, 
                              download=True, 
                              transform=transform)

### Task 3: Load the first 100 images

Due to memory constraints, create a subset of the first **100 test images** using `Subset()` from `torch.utils.data`. Wrap the subset in a PyTorch `DataLoader` with a batch size of `8` and be sure not to shuffle the test images.

In [ ]:
from torch.utils.data import Subset

num_samples = 100
subset_indices = list(range(num_samples))  
test_subset = Subset(test_data, subset_indices)
test_loader = DataLoader(test_subset, batch_size=8, shuffle=False)

### Task 4: Load the pretrained CLIP

Load the pretrained CLIP from Hugging Face under the name `"openai/clip-vit-base-patch32"` while making sure to:
- Move the model to the GPU device.
- Load its corresponding processor.

In [ ]:
from transformers import CLIPProcessor, CLIPModel
clip_model_name = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(clip_model_name).to(device)
clip_processor = CLIPProcessor.from_pretrained(clip_model_name)

---
## Task Group 2: Building and Training a Convolutional Autoencoder

### Task 5: Define the CIFARAutoencoder class

Create a convolution-based autoencoder class with an encoder, a decoder, and a forward method.

If using pretrained weights, be sure to construct the architecture exactly in the following order:

1. **Encoder**
    - Add a 2D convolutional layer that:
        - Takes `3` input RGB channels and outputs `8` channels.
        - Uses a `3x3` kernel, `stride=2`, and `padding=1`.
    - Followed by a `ReLU` activation.
    - Finish with a second 2D convolutional layer that:
        - Takes `8` input channels and outputs `16` channels.
        - Also uses a `3x3` kernel, `stride=2`, and `padding=1`.  

2. **Decoder**
    - Add a 2D transposed (reverse) convolutional layer that:
        - Takes `16` input channels and outputs `8` channels.
        - Uses a `3x3` kernel, `stride=2`, `padding=1`, and `output_padding=1`.
    - Followed by a `ReLU` activation.
    - Add a second 2D transposed (reverse) convolutional layer that:
        - Takes `8` input channels and outputs `3` channels (matching the original).
        - Uses a `3x3` kernel, `stride=2`, `padding=1`, and `output_padding=1`.
    - Finish with a `Sigmoid` activation.
      
3. **Forward Method**
    - Pass the input image through the encoder to obtain the latent representation.
    - Pass the latent representation through the decoder to obtain the reconstructed image.
    - Return the reconstructed image.

In [ ]:
torch.manual_seed(42)

class CIFARAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoder: compress 3x32x32 images
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 8, 3, stride=2, padding=1), 
            nn.ReLU(),
            nn.Conv2d(8, 16, 3, stride=2, padding=1),
        )
        
        # Decoder: reconstruct back to 3x32x32
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(16, 8, 3, stride=2, padding=1, output_padding=1),  
            nn.ReLU(),
            nn.ConvTranspose2d(8, 3, 3, stride=2, padding=1, output_padding=1),  
            nn.Sigmoid() 
        )
    
    def forward(self, x):
        latent_x = self.encoder(x)
        recon_x = self.decoder(latent_x)
        return recon_x

### Task 6: Instantiate the autoencoder

Create an instance of the autoencoder class and move it to the appropriate device.

In [ ]:
torch.manual_seed(42)

conv_autoencoder = CIFARAutoencoder().to(device)

### Train Autoencoder Task: Load the training data, define the loss function, optimizer, and training loop

**Note:** Due to memory limits in the learning environment, we recommend training the autoencoder separately (either offline or in another notebook) and saving the weights for later use.

Be sure to set a random seed for reproducibility.

In [ ]:
torch.manual_seed(42)

data_dir = '/home/ccuser/data'

transform = transforms.Compose([
    transforms.ToTensor(),
])

train_data = datasets.CIFAR10(root=data_dir, 
                               train=True, 
                               download=True, 
                               transform=transform)

train_loader = DataLoader(train_data, batch_size=8, shuffle=True)
print(f"Training samples: {len(train_data)}")

In [ ]:
torch.manual_seed(42)

criterion = nn.MSELoss()
optimizer = optim.Adam(conv_autoencoder.parameters(), lr=1e-3)
epochs = 5

print("Starting training...")
for epoch in range(epochs):
    total_loss = 0
    for x, _ in train_loader:
        x = x.to(device)
        optimizer.zero_grad()
        recon = conv_autoencoder(x)
        loss = criterion(recon, x)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

print("Training complete!")

# Save weights
# torch.save(conv_autoencoder.state_dict(), "cifar_autoencoder.pt")
print("Weights saved to cifar_autoencoder.pt")

### Task 7: Load the pretrained weights (or train the autoencoder on your own)

Load a set of pretrained autoencoder weights saved under `"cifar_autoencoder.pt"`. Use `torch.load()` with `weights_only=True` and `map_location=device`, then load the state dict into your autoencoder.

In [ ]:
# Load pretrained weights
state_dict = torch.load("cifar_autoencoder.pt", weights_only=True, map_location=device)
conv_autoencoder.load_state_dict(state_dict)

print("Loaded pretrained autoencoder successfully.")
print(conv_autoencoder)

<details><summary style="display:list-item; font-size:16px; color:blue;">(Optional) Train the autoencoder</summary>

Optionally, you can train your own autoencoder by running the training loop below. This will train the autoencoder to reconstruct the CIFAR-10 images from scratch. Due to memory limits in the learning environment, we recommend training the autoencoder separately (either offline or in another notebook) and saving the weights for later use:

```py
# Train autoencoder 
torch.manual_seed(42)
epochs = 5
print("Starting training...")
for epoch in range(epochs):
    total_loss = 0
    for x, _ in train_loader:
        x = x.to(device)
        optimizer.zero_grad()
        recon = conv_autoencoder(x)
        loss = criterion(recon, x)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

print("Training complete!")

# Save trained weights
torch.save(conv_autoencoder.state_dict(), "cifar_autoencoder.pt")
print("Weights saved to cifar_autoencoder.pt")
```
<br>

We've provided a notebook that trains an autoencoder to obtain our pretrained weights. To access the directory containing the notebook, click on the Jupyter icon in the top-left corner of the current notebook.

</details>

### Task 8: Set the autoencoder to evaluation mode

Set the autoencoder to evaluation mode using `.eval()` to disable dropout and batch normalization training behavior.

In [ ]:
conv_autoencoder.eval()

### Task 9: Reconstruct the first 100 images in the testing dataset

Pass the 100 test images (in batches) through the autoencoder to generate reconstructed versions of each image.
- Store the batch of reconstructed images.
- Store the true labels. 

In [ ]:
batched_recons = []
true_labels = []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        recon = conv_autoencoder(x)   
        batched_recons.append(recon.cpu())
        true_labels.extend(y.tolist())

### Task 10: Concatenate all reconstructed batches

Concatenate all reconstructed batches into a single PyTorch tensor using `torch.cat()` with `dim=0`.

In [ ]:
all_recons = torch.cat(batched_recons, dim=0)
print("All reconstructions `all_recons` shape:", all_recons.shape)

---
## Task Group 3: CLIP Zero-Shot Classification

### Task 11: Create text descriptions for each CIFAR10 class

Generate a list of text descriptions (one for each CIFAR-10 class) that CLIP will use as zero-shot labels. Format each description as `"a photo of a {class_name}"`.

In [ ]:
text_descriptions = [f"a photo of a {name}" for name in cifar10_classes]

print(f"Text descriptions: {text_descriptions}")

### Task 12: Define a function to perform zero-shot classification

Create a function that performs zero-shot classification with the CLIP model. The function should take in the reconstructed or original image (as a tensor), the list of text descriptions, the CLIP model, and the CLIP processor:
- Use `transforms.ToPILImage()(image_tensor)` to conver the image tensor `image_tensor` into PIL images that CLIP expects.
- Pass the PIL images through the processor into preprocessed inputs.
- Pass the preprocessed inputs through the CLIP model.
- Obtain the raw outputs, similarity scores, and convert the scores to probabilities.
- The function should return the similarity scores and their probabilities.


In [ ]:
from torchvision import transforms

def zero_shot_classification(image_tensor, text_descriptions, model, processor):
    with torch.inference_mode():
        image_pil = transforms.ToPILImage()(image_tensor)
        
        inputs = processor(
            text=text_descriptions,
            images=image_pil,
            return_tensors="pt",
            padding=True
        )

        # Detect device and move inputs to that device
        device = next(model.parameters()).device
        inputs = {k: v.to(device) for k, v in inputs.items()}

        outputs = model(**inputs)
        scores = outputs.logits_per_image[0]
        probs = scores.softmax(-1)
        return scores, probs

### Task 13: Run zero-shot classification on the reconstructed images

Use the zero-shot classification function to predict labels for each reconstructed CIFAR-10 image.

For each reconstructed image: 
- Compute CLIP's class probabilities.
- Select the class with the highest probability.
- Store the predicted label.

In [ ]:
recon_pred = []
for image in all_recons:
    scores, probs = zero_shot_classification(image, text_descriptions, clip_model, clip_processor)
    pred = probs.argmax().item()
    recon_pred.append(pred)

### Task 14: Run zero-shot classification on the original images

Use the zero-shot classification function to predict labels for each original CIFAR-10 image.

For each original image: 
- Compute CLIP's class probabilities.
- Select the class with the highest probability.
- Store the predicted label.

In [ ]:
orig_pred = []
for image, label in test_subset:
    scores, probs = zero_shot_classification(image, text_descriptions, clip_model, clip_processor)
    pred = probs.argmax().item()
    orig_pred.append(pred)

---
## Task Group 4: Comparing Classification Performance

### Task 15: Compare the accuracy between reconstructed and original images

Evaluate CLIP's zero-shot classification performance by computing the *overall* accuracy for both the reconstructed and original images.

Print both accuracy scores.

In [ ]:
from sklearn.metrics import accuracy_score

recon_acc = accuracy_score(true_labels, recon_pred)
orig_acc = accuracy_score(true_labels, orig_pred)

print("Accuracy Score (Reconstructed Images):", recon_acc)
print("Accuracy Score (Original Images):", orig_acc)

- Is accuracy a meaningful performance metric considering that there are ten CIFAR-10 classes?

### Task 16: Generate classification reports

Create full classification reports for both the reconstructed and original images showing the *per-class* precision, recall, and F1-scores. 

Print both classification reports. 

In [ ]:
from sklearn.metrics import classification_report

recon_report = classification_report(true_labels, recon_pred, target_names=cifar10_classes)
orig_report = classification_report(true_labels, orig_pred, target_names=cifar10_classes)

print("\n=== Classification Report (Reconstructed Images) ===")
print(recon_report)

print("\n=== Classification Report (Original Images) ===")
print(orig_report)

### Task 17: Compare heatmaps between original and reconstructed images

Visualize the classification reports as heatmaps for the reconstructed and original images:
- Re-compute each classification report as a dictionary using `output_dict=True`. Then, convert the report to a pandas DataFrame and transpose it so that each row corresponds to a CIFAR-10 class with columns for their precision, recall, and F1-score.
- Use seaborn to create heatmaps for each report to compare CLIP's performance on reconstructed versus original images. 

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

recon_report_dict = classification_report(true_labels, recon_pred, target_names=cifar10_classes, output_dict=True)
recon_report_transpose = pd.DataFrame(recon_report_dict).transpose()
recon_metrics = recon_report_transpose.loc[cifar10_classes, ["precision", "recall", "f1-score"]]

orig_report_dict = classification_report(true_labels, orig_pred, target_names=cifar10_classes, output_dict=True)
orig_report_transpose = pd.DataFrame(orig_report_dict).transpose()
orig_metrics = orig_report_transpose.loc[cifar10_classes, ["precision", "recall", "f1-score"]]

plt.figure(figsize=(10, 6))
sns.heatmap(recon_metrics, annot=True, cmap="Blues", fmt=".2f")
plt.title("CLIP Classification Report Heatmap (Original Images)")
plt.show()

plt.figure(figsize=(10, 6))
sns.heatmap(orig_metrics, annot=True, cmap="Blues", fmt=".2f")
plt.title("CLIP Classification Report Heatmap (Original Images)")
plt.show()

- For the **original images**, which classes does CLIP perform well on? How can you tell from the heatmap?
- For the **reconstructed images**, which classes show noticeably lower precision, recall, or F1-score?

### Task 18: Compute the per-class metric differences (precision, recall, and F1-score)

Create a DataFrame that shows the difference in precision, recall, and F1-scores between original and reconstructed images for each class.

In [ ]:
metric_diff = orig_metrics - recon_metrics
metric_diff

- Which class experienced the most significant drop in performance after reconstruction?
- Are there any classes where performance stays relatively strong even after reconstruction?
- Are there specific metrics (precision, recall, or F1-score) that degrade more noticeably across classes?

### Task 19: Visualize original and reconstructed images

Create a side-by-side visualization comparing the first 10 CIFAR-10 test original images and their reconstructed versions.

In [ ]:
n = 10
plt.figure(figsize=(15, 4))
for i in range(n):
    # original images
    plt.subplot(2, n, i+1)
    plt.imshow(test_data[i][0].permute(1,2,0))
    plt.axis("off")
    
    # reconstructed images
    plt.subplot(2, n, n+i+1)
    plt.imshow(all_recons[i].permute(1,2,0))
    plt.axis("off")

plt.suptitle("Original (Top) vs Reconstructed (Bottom)")
plt.show()

How well did the autoencoder preserve key image features?

---
## Conclusion

Congratulations on completing the project. You successfully investigated how the image reconstruction quality of autoencoders affects the zero-shot classification performance of a pretrained CLIP multimodal model on the CIFAR-10 dataset!

In this project, you learned how to:
1. Load and prepare image datasets.
2. Build and train a convolutional autoencoder to compress and reconstruct images.
4. Apply pretrained multimodal models (CLIP) for zero-shot classification.
5. Compare model performance with visualizations.

**Overall accuracy performance drop**
- The pretrained CLIP achieves a 91% accuracy on the original images and only 77% accuracy on the reconstructed images. This 14% differential shows that even a small autoencoder introduces enough visual distortion to reduce CLIP's zero-shot classification performance!

**Per-class observations**
- The classes that showed the most significant drop in performance were automobile, bird, cat, and truck. This suggests that their finer texture and details were not fully preserved by the autoencoder.
- The classes that showed the least significant drop in performance were ship, frog, airplane, and horse. This suggests that their broad shapes or color patterns remain recognizable after reconstruction.


### Takeaways

This experiment highlights a key insight: zero-shot models like CLIP are extremely powerful, but their performance depends heavily on the quality of the input images. Small reconstructions that distort an image can have a measurable impact!

Most importantly, you now have the workflow tools to design and test similar experiments on your own real-world dataset and tasks. Keep going - you're well on your way to building real-world AI projects! 

Happy coding!